# Preprocessing — Member 1: Feature Engineering & Outlier/Skew Treatment

## Online Shopper Purchase Likelihood Prediction System

**IT3051 – Fundamentals of Data Mining | Mini Project 2026 | Group DS_WE_01.02**

This notebook covers the second step in the preprocessing relay, building on Member 4's
train/test split. It derives new features (`TotalPages`, `TotalDuration`, `AvgTimePerPage`,
`VisitorType_Weekend`) from raw training/test values, then applies a log1p transform to
correct right-skewed features identified during EDA.

In [1]:
import sys
import os

# Add the project root (one level up from /notebooks) to Python's search path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import pandas as pd
from src.preprocessing.feature_engineering import (
    add_engineered_features,
    apply_log_transform,
    SKEWED_COLUMNS,
)

# --- Loading Member 4's already-split, deduplicated data ---

In [3]:
X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")

# --- Step A: Feature engineering (raw values, before any transform) ---

We engineered TotalPages, TotalDuration, and AvgTimePerPage to give the model an overall measure of browsing volume and engagement pace, rather than requiring it to infer this from three separate page-count/duration columns. We also created VisitorType_Weekend, motivated by our EDA finding that the weekend effect on purchase likelihood differs meaningfully between new and returning visitors — this interaction wasn't captured by either variable alone.

In [4]:
X_train = add_engineered_features(X_train)
X_test = add_engineered_features(X_test)

# --- Step B: Checking skew of the two NEW derived columns before deciding to transform them ---

Since TotalPages and TotalDuration are sums of already-skewed raw columns, we checked their skewness rather than assuming they'd need the same treatment. Both came back strongly skewed (4.24 and 8.19 respectively), so we extended the log1p transformation to include them.

In [5]:
print("Skew check on derived features:")
print(X_train[["TotalPages", "TotalDuration"]].skew())

Skew check on derived features:
TotalPages       4.239776
TotalDuration    8.187805
dtype: float64


In [6]:
columns_to_transform = SKEWED_COLUMNS + ["TotalPages", "TotalDuration"]

# --- Step C: Applying log1p transform, in place ---

We applied log1p in place (not as separate _log columns) since it's a monotonic transform — keeping both versions would create near-duplicate, highly correlated columns with no benefit for any of our planned models. This transform matters for distance/coefficient-based models (Logistic Regression, SVM, KNN); for tree-based models it's largely inert since splits are rank-based, but applying it uniformly keeps the pipeline simple.

In [7]:
X_train = apply_log_transform(X_train, columns=columns_to_transform)
X_test = apply_log_transform(X_test, columns=columns_to_transform)

After applying log1p, TotalPages became nearly symmetric (skew: 4.24 → -0.11). TotalDuration improved substantially in magnitude (8.19 → -1.53) but overcorrected into mild left-skew, likely due to the large cluster of near-zero values in its underlying duration columns compressing disproportionately under the log transform. We retained the log-transformed version regardless, since -1.53 is a substantial improvement over the untransformed distribution and remains suitable for the distance/coefficient-based models in our comparison.

In [8]:
print("Skew AFTER log1p transform:")
print(X_train[["TotalPages", "TotalDuration"]].skew())

Skew AFTER log1p transform:
TotalPages      -0.106369
TotalDuration   -1.532358
dtype: float64


# --- Step D: Sanity checking the output ---

In [9]:
print("\nShape check:", X_train.shape, X_test.shape)
print("\nSample of transformed + engineered columns:")
print(X_train[["ProductRelated", "PageValues", "TotalPages", "TotalDuration",
                "AvgTimePerPage", "VisitorType_Weekend"]].head())


Shape check: (9764, 21) (2441, 21)

Sample of transformed + engineered columns:
   ProductRelated  PageValues  TotalPages  TotalDuration  AvgTimePerPage  \
0        3.761200    3.646944    3.951244       7.136483       24.627451   
1        3.433987    3.559580    3.433987       6.804935       30.042963   
2        2.639057    0.000000    2.639057       6.045400       32.397436   
3        1.791759    0.000000    2.079442       6.396096       85.500000   
4        2.833213    0.000000    2.833213       7.377926       99.954167   

       VisitorType_Weekend  
0  Returning_Visitor_False  
1        New_Visitor_False  
2  Returning_Visitor_False  
3         New_Visitor_True  
4  Returning_Visitor_False  


In [10]:
print("\nX_test sample (same check, other dataset):")
print(X_test[["ProductRelated", "PageValues", "TotalPages", "TotalDuration",
              "AvgTimePerPage", "VisitorType_Weekend"]].head())
print("\nX_test shape:", X_test.shape)


X_test sample (same check, other dataset):
   ProductRelated  PageValues  TotalPages  TotalDuration  AvgTimePerPage  \
0        2.564949         0.0    2.564949       6.181051       40.208333   
1        2.833213         0.0    2.944439       6.635268       42.249074   
2        3.401197         0.0    3.583519       7.781570       68.429524   
3        2.484907         0.0    2.944439       6.415015       33.886111   
4        1.791759         0.0    1.791759       0.000000        0.000000   

       VisitorType_Weekend  
0  Returning_Visitor_False  
1         New_Visitor_True  
2  Returning_Visitor_False  
3  Returning_Visitor_False  
4  Returning_Visitor_False  

X_test shape: (2441, 21)


# --- Step E: Saving output for Member 2 / Member 3 to pick up next ---

In [11]:
X_train.to_csv("../data/processed/X_train_engineered.csv", index=False)
X_test.to_csv("../data/processed/X_test_engineered.csv", index=False)
print("\nSaved to data/processed/")


Saved to data/processed/
